# Advanced IR: lightning-ir (Part 2)

In this part of the tutorial, you will get to know another deep learning library, this time one that uses the lightning library to implement several well-known neural IR models.

We will use one of these neural models to see how well it ranks documents on the test collection from last week and a new test collection.

As with Part 1, if you do not have access to a computer with a GPU, it is strongly suggested that you run this notebook in Google Colab. Also make sure you upload the `lightning_ir_pyterrier.py` file into Colab for everything to work. 

### Preparation: Install dependencies

First, we install the libraries

In [1]:
!pip install python-terrier lightning_ir pyterrier-alpha tqdm

In [13]:
dataset_name = "nano-beir/msmarco"

In [14]:
import os
from pathlib import Path
import torch

from lightning_ir import TorchDenseIndexConfig, TorchDenseSearchConfig
from lightning_ir_pyterrier import BiEncoder

from pyterrier.datasets import get_dataset
import pyterrier as pt

from pyterrier.datasets import get_dataset
dataset = get_dataset(f"irds:{dataset_name}")

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
if torch.backends.mps.is_available():
    device = "mps"

In [15]:
print(device)

cuda


In the next cell, we set up the neural ranker. You don't need to know what the model is doing yet, but that it will offload some computation to make it more efficient at inference time, and this is what the vector index is.

In [16]:
index_dir = Path(f"./vector_index_{dataset_name.replace('/','-')}")

neural_ranker = BiEncoder(
    model_name_or_path="webis/bert-bi-encoder",
    index_dir=index_dir,
    index_config=TorchDenseIndexConfig(),
    search_config=TorchDenseSearchConfig(k=100),
    batch_size=32,
    verbose=True,
    use_gpu=True
)
neural_ranker._module.to(device)
neural_ranker._module.device # (if you run on a GPU, it should say so!)

You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BertTokenizer'. 
The class this function is called from is 'LirDprBertTokenizerFast'.


device(type='cuda', index=0)

### Step 1: Indexing

The next step we will pre-process the documents to make query processing faster.

In [17]:
from pathlib import Path
if not index_dir.exists():
    neural_ranker.index(dataset.get_corpus_iter())


[INFO] [starting] building docstore
[INFO] [starting] https://huggingface.co/datasets/zeta-alpha-ai/NanoMSMARCO/resolve/main/corpus/train-00000-of-00001.parquet
                                                                                
                                                                                                                                                  [INFO] [finished] https://huggingface.co/datasets/zeta-alpha-ai/NanoMSMARCO/resolve/main/corpus/train-00000-of-00001.parquet: [00:00] [1.12MB] [6.72MB/s]
docs_iter: 100%|█████████████████████████| 5043/5043 [00:00<00:00, 5113.59doc/s]
[INFO] [finished] docs_iter: [00:00] [5043doc] [5112.03doc/s]
[INFO] [finished] building docstore s]
nano-beir/msmarco documents: 100%|██████████| 5043/5043 [00:03<00:00, 1343.08it/s]
webis/bert-bi-encoder index: 158batch [00:03, 41.93batch/s]


### Step 2: Search

Next, we can use the model to query this vector index.

In [18]:
neural_ranker.search("0 dimensional biomaterials show inductive properties")

webis/bert-bi-encoder search: 100%|██████████| 1/1 [00:00<00:00, 173.84batch/s]


,index,qid,query,docno,score,rank
0,0,1,0 dimensional biomaterials show inductive prop...,1963339,25.146534,0
1,0,1,0 dimensional biomaterials show inductive prop...,6988663,24.983438,1
2,0,1,0 dimensional biomaterials show inductive prop...,1332664,24.455074,2
3,0,1,0 dimensional biomaterials show inductive prop...,2584299,24.360256,3
4,0,1,0 dimensional biomaterials show inductive prop...,5351437,24.204706,4
...,...,...,...,...,...,...
95,0,1,0 dimensional biomaterials show inductive prop...,3756781,21.652239,95
96,0,1,0 dimensional biomaterials show inductive prop...,5414327,21.634129,96
97,0,1,0 dimensional biomaterials show inductive prop...,689736,21.607664,97
98,0,1,0 dimensional biomaterials show inductive prop...,1690952,21.579639,98


Since we want to compare the model to the models from last week, we need to index the data in pyterrier too.

In [19]:
pt_index_path = f"./index_{dataset_name.replace('/','-')}"

if not os.path.exists(pt_index_path + "/data.properties"):
  # create the index, using the IterDictIndexer indexer 
  indexer = pt.index.IterDictIndexer(pt_index_path, meta_reverse=[])

  # we give the dataset get_corpus_iter() directly to the indexer
  # while specifying the fields to index and the metadata to record
  index_ref = indexer.index(dataset.get_corpus_iter())

else:
  # if you already have the index, use it.
  index_ref = pt_index_path

index = pt.IndexFactory.of(index_ref)

tfidf = pt.BatchRetrieve(index, wmodel="TF_IDF")
bm25 = pt.BatchRetrieve(index, wmodel="BM25")

nano-beir/msmarco documents: 100%|██████████| 5043/5043 [00:00<00:00, 27423.56it/s]
/tmp/ipykernel_113309/3690260414.py:17: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  tfidf = pt.BatchRetrieve(index, wmodel="TF_IDF")
/tmp/ipykernel_113309/3690260414.py:18: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25 = pt.BatchRetrieve(index, wmodel="BM25")


### Step 3: Evaluation

Now, we can see some results, just like last week.

In [20]:
from pandas import DataFrame

topics = dataset.get_topics(variant="text").iloc[:3]
neural_ranker.transform(topics)

[INFO] [starting] https://huggingface.co/datasets/zeta-alpha-ai/NanoMSMARCO/resolve/main/queries/train-00000-of-00001.parquet
[INFO] [finished] https://huggingface.co/datasets/zeta-alpha-ai/NanoMSMARCO/resolve/main/queries/train-00000-of-00001.parquet: [00:00] [3.22kB] [55.6MB/s]
webis/bert-bi-encoder search: 100%|██████████| 1/1 [00:00<00:00, 130.84batch/s]                                                         


,index,qid,query,docno,score,rank
0,0,994479,which health care system provides all citizens...,7837267,32.170685,0
1,0,994479,which health care system provides all citizens...,575979,32.007507,1
2,0,994479,which health care system provides all citizens...,8083127,31.621677,2
3,0,994479,which health care system provides all citizens...,1674646,31.553629,3
4,0,994479,which health care system provides all citizens...,615746,30.920576,4
...,...,...,...,...,...,...
295,2,1088332,weather in oran,7114600,22.521492,95
296,2,1088332,weather in oran,5155867,22.438265,96
297,2,1088332,weather in oran,6454998,22.419092,97
298,2,1088332,weather in oran,5212100,22.418304,98


And we can compare the three retrieval models against each other.

In [21]:
from ir_measures import nDCG, RR, MAP
from pyterrier.pipelines import Experiment
from pyterrier.text import get_text

Experiment(
    [
        tfidf,
        bm25,
        neural_ranker,
    ],
    dataset.get_topics(variant='text'),
    dataset.get_qrels(),
    eval_metrics=[nDCG@10, MAP, RR],
    names=[
        "tf-idf",
        "BM25",
        "neural-ranker",
    ],
    verbose=True,
)

[INFO] [starting] https://huggingface.co/datasets/zeta-alpha-ai/NanoMSMARCO/resolve/main/qrels/train-00000-of-00001.parquet
[INFO] [finished] https://huggingface.co/datasets/zeta-alpha-ai/NanoMSMARCO/resolve/main/qrels/train-00000-of-00001.parquet: [00:00] [2.10kB] [26.3MB/s]
pt.Experiment: 100%|██████████| 3/3 [00:00<00:00, 11.23system/s]


,name,nDCG@10,AP,RR
0,tf-idf,0.505932,0.448317,0.448317
1,BM25,0.505932,0.448514,0.448514
2,neural-ranker,0.622145,0.568540,0.568540


In [12]:
from ir_measures import nDCG, RR, MAP
from pyterrier.pipelines import Experiment
from pyterrier.text import get_text

Experiment(
    [
        tfidf,
        bm25,
        neural_ranker,
    ],
    dataset.get_topics(variant='text'),
    dataset.get_qrels(),
    eval_metrics=[nDCG@10, MAP, RR],
    names=[
        "tf-idf",
        "BM25",
        "neural-ranker",
    ],
    verbose=True,
)

pt.Experiment: 100%|██████████| 3/3 [00:05<00:00,  1.77s/system]


,name,nDCG@10,AP,RR
0,tf-idf,0.665635,0.619427,0.630251
1,BM25,0.672167,0.626997,0.636829
2,neural-ranker,0.558310,0.519872,0.531023


### Step 4: MS MARCO

Maybe it's no suprise that the neural ranking model is not effective at retrieving documents for a collection it was not trained on. So what if we evaluate it on the types of queries and documents that the model actually has seen?

MS MARCO (https://microsoft.github.io/msmarco/) is one of the best resources for training effective ranking models. It contains about 8.8 million passages of documents (about a paragraph in length each), and around 100K queries with on average one known relevant document per query. This dataset was used to train the neural model above.

Let's look at how well the same models from above do on a significantly smaller, curated sample of MS MARCO with roughly the same number of documents as scifact. Change the `dataset_name` variable in the second cell to `nano-beir/msmarco`, then re-run the cells above to get the new evaluation results. Do you notice a difference?

Spend some time looking at the results you get back from these two datasets, the queries and the documents retrieved. Do the results really look much better than BM25? How might you quantify if one system is likely to be better that another system, and not just due to random chance?